# Customer Churn Analysis
### SQL, Statistics, Machine Learning & Professional Publishing

**Dataset:** IBM Telco Customer Churn  
**Project:** Integrated Assignment — Customer Churn Analysis

**Objective:** Understand which customers are likely to churn and provide data-driven recommendations to reduce churn.


## 1. Project Overview

This notebook follows all four parts of the assignment:

1. SQL queries using SQLite
2. Statistical analysis and hypothesis testing
3. Machine learning models and evaluation
4. Professional communication and GitHub portfolio

The assignment requires the IBM Telco Customer Churn dataset, an 80/20 train-test split with `random_state=42`, Logistic Regression, Decision Tree (`max_depth=4`), and Random Forest (100 trees).


## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency, ttest_ind

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

pd.set_option("display.max_columns", None)


## 3. Load the Dataset

In [ ]:
# Load the dataset with both Internet and Offline support.
import os
import pandas as pd

# Online dataset URL
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

# Local CSV filename (keep this file in the same folder as the notebook)
local_file = "Telco-Customer-Churn.csv"

try:
    # Try Internet first
    df = pd.read_csv(url)
    print("✅ Dataset loaded from Internet.")

    # Save a local copy so the notebook can work offline later
    df.to_csv(local_file, index=False)
    print("✅ Local copy saved for offline use.")

except Exception as e:
    # If Internet is unavailable, use the local CSV
    if os.path.exists(local_file):
        df = pd.read_csv(local_file)
        print("⚠️ Internet unavailable.")
        print("✅ Dataset loaded from local CSV.")
    else:
        raise FileNotFoundError(
            "Internet is unavailable and the local CSV file was not found. "
            "Please download Telco-Customer-Churn.csv and place it in the same folder as this notebook."
        )

print("Dataset shape:", df.shape)
display(df.head())


## 4. Initial Data Inspection

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isna().sum().to_frame("missing_values"))

print("\nDuplicate rows:", df.duplicated().sum())


## 5. Data Cleaning

In [ ]:
# TotalCharges contains blank strings in the original dataset.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Missing TotalCharges before imputation:",
      df["TotalCharges"].isna().sum())

# Assignment requirement: fill missing TotalCharges with the median.
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

print("Missing TotalCharges after imputation:",
      df["TotalCharges"].isna().sum())

display(df.describe().T)


## 6. Basic Churn Overview

In [ ]:
churn_counts = df["Churn"].value_counts()
churn_rate = (df["Churn"] == "Yes").mean() * 100

print("Total customers:", len(df))
print("Churned customers:", (df["Churn"] == "Yes").sum())
print(f"Overall churn rate: {churn_rate:.2f}%")

display(churn_counts.to_frame("customer_count"))

plt.figure(figsize=(7, 5))
churn_counts.plot(kind="bar")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


# Part 1 — SQL Queries

The CSV is loaded into a SQLite database and each required query is executed with `pd.read_sql_query()`.

A separate file named `customer_churn_queries.sql` contains all required SQL statements.


In [ ]:
# Create SQLite database and load the dataframe.
conn = sqlite3.connect("customer_churn.db")
df.to_sql("customers", conn, if_exists="replace", index=False)

print("SQLite database created and customers table loaded.")


### Customer Overview — Total Customers

In [ ]:
query = """SELECT COUNT(*) AS total_customers
 FROM customers;"""
result = pd.read_sql_query(query, conn)
display(result)


### Customer Overview — Churned Customers

In [ ]:
query = """SELECT COUNT(*) AS churned_customers
 FROM customers
 WHERE Churn = 'Yes';"""
result = pd.read_sql_query(query, conn)
display(result)


### Demographics — Customers by Gender

In [ ]:
query = """SELECT gender, COUNT(*) AS customer_count
 FROM customers
 GROUP BY gender;"""
result = pd.read_sql_query(query, conn)
display(result)


### Demographics — Senior Citizens

In [ ]:
query = """SELECT COUNT(*) AS senior_citizens
 FROM customers
 WHERE SeniorCitizen = 1;"""
result = pd.read_sql_query(query, conn)
display(result)


### Contract & Tenure — Average Tenure by Churn

In [ ]:
query = """SELECT Churn, AVG(tenure) AS average_tenure
 FROM customers
 GROUP BY Churn;"""
result = pd.read_sql_query(query, conn)
display(result)


### Contract & Tenure — Customers by Contract

In [ ]:
query = """SELECT Contract, COUNT(*) AS customer_count
 FROM customers
 GROUP BY Contract;"""
result = pd.read_sql_query(query, conn)
display(result)


### Internet Service & Charges — Average Monthly Charges

In [ ]:
query = """SELECT InternetService,
        AVG(MonthlyCharges) AS average_monthly_charges
 FROM customers
 GROUP BY InternetService;"""
result = pd.read_sql_query(query, conn)
display(result)


### Internet Service & Charges — Total Charges by Payment Method

In [ ]:
query = """SELECT PaymentMethod,
        SUM(TotalCharges) AS total_charges
 FROM customers
 GROUP BY PaymentMethod;"""
result = pd.read_sql_query(query, conn)
display(result)


### Churn Drivers — Churn Rate by Contract

In [ ]:
query = """SELECT Contract,
        AVG(CASE WHEN Churn='Yes' THEN 1.0 ELSE 0.0 END)*100
          AS churn_rate_percent
 FROM customers
 GROUP BY Contract
 ORDER BY churn_rate_percent DESC;"""
result = pd.read_sql_query(query, conn)
display(result)


### Churn Drivers — Churn Rate by Internet Service

In [ ]:
query = """SELECT InternetService,
        AVG(CASE WHEN Churn='Yes' THEN 1.0 ELSE 0.0 END)*100
          AS churn_rate_percent
 FROM customers
 GROUP BY InternetService
 ORDER BY churn_rate_percent DESC;"""
result = pd.read_sql_query(query, conn)
display(result)


### Churn Drivers — Churn Rate by Online Security

In [ ]:
query = """SELECT OnlineSecurity,
        AVG(CASE WHEN Churn='Yes' THEN 1.0 ELSE 0.0 END)*100
          AS churn_rate_percent
 FROM customers
 GROUP BY OnlineSecurity
 ORDER BY churn_rate_percent DESC;"""
result = pd.read_sql_query(query, conn)
display(result)


### Advanced Aggregation — Top 5 Segments

In [ ]:
query = """SELECT Contract, InternetService,
        COUNT(*) AS customers,
        AVG(CASE WHEN Churn='Yes' THEN 1.0 ELSE 0.0 END)*100
          AS churn_rate_percent
 FROM customers
 GROUP BY Contract, InternetService
 ORDER BY churn_rate_percent DESC
 LIMIT 5;"""
result = pd.read_sql_query(query, conn)
display(result)


## SQL Interpretation

Use the displayed DataFrames to identify:
- the overall size of the customer base;
- the number and proportion of churned customers;
- demographic composition;
- differences in tenure;
- contract and internet-service patterns;
- payment-method concentration;
- the highest-risk contract/service segments.

The most useful business signal should be carried forward into the statistical and ML sections.


# Part 2 — Statistical Analysis & Hypothesis Testing

## 7. Descriptive Statistics

In [ ]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

descriptive = df[numeric_cols].agg(["mean", "median", "std"]).T
descriptive["skewness"] = df[numeric_cols].skew()

display(descriptive)


### Interpretation Guide

- If mean and median are close, the distribution is relatively balanced.
- If the mean is noticeably higher than the median, the variable may be right-skewed.
- A large standard deviation indicates greater spread.
- Skewness values farther from zero indicate stronger asymmetry.

Use the actual values produced above in the final submitted interpretation.


## 8. Chi-Square Test — Contract vs Churn

**H0:** Contract type and Churn are independent.  
**H1:** Contract type and Churn are associated.

Significance level: **α = 0.05**


In [ ]:
contingency_table = pd.crosstab(df["Contract"], df["Churn"])
display(contingency_table)

chi2_stat, chi2_p, dof, expected = chi2_contingency(contingency_table)

print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"p-value: {chi2_p:.10g}")

if chi2_p < 0.05:
    print("Decision: Reject H0.")
    print("Interpretation: Contract type is significantly associated with Churn.")
else:
    print("Decision: Fail to reject H0.")
    print("Interpretation: There is insufficient evidence of an association.")


## 9. Independent T-Test — MonthlyCharges

**H0:** The mean MonthlyCharges are equal for churned and non-churned customers.  
**H1:** The mean MonthlyCharges differ between the two groups.

A Welch independent-samples t-test is used because it does not require equal variances.


In [ ]:
monthly_churned = df.loc[df["Churn"] == "Yes", "MonthlyCharges"]
monthly_nonchurned = df.loc[df["Churn"] == "No", "MonthlyCharges"]

t_stat, t_p = ttest_ind(
    monthly_churned,
    monthly_nonchurned,
    equal_var=False
)

print(f"Churned mean MonthlyCharges: {monthly_churned.mean():.2f}")
print(f"Non-churned mean MonthlyCharges: {monthly_nonchurned.mean():.2f}")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {t_p:.10g}")

if t_p < 0.05:
    print("Decision: Reject H0.")
    print("Interpretation: MonthlyCharges differ significantly between the groups.")
else:
    print("Decision: Fail to reject H0.")
    print("Interpretation: There is insufficient evidence of a difference.")


## 10. Statistical Business Recommendation

Based on the hypothesis tests, write one clear recommendation. A strong recommendation should connect the statistically significant variable(s) to a practical retention action.

**Suggested structure:**  
“Because [finding] is statistically significant, the company should [specific action] for [specific customer group].”


# Part 3 — Machine Learning

## 11. Prepare Features and Target

In [ ]:
# Drop customerID because it is an identifier, not a useful predictive feature.
X = df.drop(columns=["customerID", "Churn"]).copy()

# Map target to 0/1 as required.
y = df["Churn"].map({"No": 0, "Yes": 1})

# One-hot encode categorical variables.
X = pd.get_dummies(X, drop_first=True)

print("Feature matrix shape:", X.shape)
print("Target distribution:")
display(y.value_counts().rename(index={0: "No Churn", 1: "Churn"}).to_frame("count"))


## 12. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 13. Baseline — Logistic Regression

In [ ]:
# Standardize features for Logistic Regression.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logistic = LogisticRegression(max_iter=2000, random_state=42)
logistic.fit(X_train_scaled, y_train)

logistic_pred = logistic.predict(X_test_scaled)

print(classification_report(
    y_test,
    logistic_pred,
    target_names=["No Churn", "Churn"]
))

print("Confusion Matrix:")
display(pd.DataFrame(
    confusion_matrix(y_test, logistic_pred),
    index=["Actual No Churn", "Actual Churn"],
    columns=["Predicted No Churn", "Predicted Churn"]
))


## 14. Decision Tree — max_depth=4

In [ ]:
tree = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

tree.fit(X_train, y_train)
tree_pred = tree.predict(X_test)

print(classification_report(
    y_test,
    tree_pred,
    target_names=["No Churn", "Churn"]
))

print("Confusion Matrix:")
display(pd.DataFrame(
    confusion_matrix(y_test, tree_pred),
    index=["Actual No Churn", "Actual Churn"],
    columns=["Predicted No Churn", "Predicted Churn"]
))


## 15. Random Forest — 100 Trees

In [ ]:
random_forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

random_forest.fit(X_train, y_train)
rf_pred = random_forest.predict(X_test)

print(classification_report(
    y_test,
    rf_pred,
    target_names=["No Churn", "Churn"]
))

print("Confusion Matrix:")
display(pd.DataFrame(
    confusion_matrix(y_test, rf_pred),
    index=["Actual No Churn", "Actual Churn"],
    columns=["Predicted No Churn", "Predicted Churn"]
))


## 16. Model Comparison

In [ ]:
def evaluate_model(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0)
    }

model_results = pd.DataFrame([
    evaluate_model("Logistic Regression", y_test, logistic_pred),
    evaluate_model("Decision Tree", y_test, tree_pred),
    evaluate_model("Random Forest", y_test, rf_pred)
])

display(model_results.sort_values("F1", ascending=False))


### Model Selection

Do not choose the model using accuracy alone. For churn prediction, **recall** and **F1-score** are important because missing a customer who is likely to churn can reduce the value of a retention campaign.

Use the comparison table to identify the best model for the assignment.


## 17. Random Forest Feature Importance

In [ ]:
feature_importance = (
    pd.Series(random_forest.feature_importances_, index=X.columns)
      .sort_values(ascending=False)
)

display(feature_importance.head(15).to_frame("importance"))

plt.figure(figsize=(10, 6))
feature_importance.head(15).sort_values().plot(kind="barh")
plt.title("Top 15 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## 18. Model Interpretation

Compare the most important Random Forest features with the earlier SQL and statistical findings.

Questions to answer:

1. Which features are most predictive of churn?
2. Is Contract among the strongest predictors?
3. Do tenure and MonthlyCharges appear important?
4. Does the ML model support the statistical finding about Contract and Churn?
5. Which customer groups should receive retention attention first?


## 19. Final Business Recommendation

**Recommended strategy:** Build a churn-risk workflow around the best-performing model. Prioritize high-risk customers, especially groups identified by contract, tenure, service configuration, and monthly-charge patterns. Use targeted retention offers rather than a one-size-fits-all campaign.

Before deployment, monitor recall/F1, false negatives, customer response, and retention cost to ensure the model produces business value.


# Part 4 — Professional Communication & GitHub Portfolio

## Recommended repository

`customer-churn-analysis`

### Suggested structure

```text
customer-churn-analysis/
├── customer_churn_analysis.ipynb
├── customer_churn_queries.sql
├── README.md
├── customer_churn.db
└── screenshots-or-charts/
```

The assignment asks for a public GitHub repository containing the notebook, SQL file, README, and optionally the cleaned dataset.


## 20. README Content

The repository README should contain:

### Project Title
Customer Churn Analysis

### Business Problem
Identify customers likely to churn and provide data-driven recommendations to reduce churn.

### Data Source
IBM Telco Customer Churn dataset.

### Approach
- SQL analysis with SQLite
- Descriptive statistics
- Chi-square hypothesis testing
- Independent t-test
- Logistic Regression
- Decision Tree
- Random Forest
- Feature importance analysis

### Tools
Python, Pandas, NumPy, SQLite, SciPy, Scikit-learn, Matplotlib, Jupyter Notebook.

### How to Run
1. Install Python 3.
2. Install the required packages.
3. Open the notebook in Jupyter or VS Code.
4. Run cells from top to bottom.
5. The notebook downloads the dataset directly from the source URL.


# Final Submission Checklist

- [ ] `customer_churn_analysis.ipynb`
- [ ] `customer_churn_queries.sql`
- [ ] `README.md`
- [ ] SQLite database generated by the notebook
- [ ] SQL DataFrame outputs displayed
- [ ] Descriptive statistics completed
- [ ] Chi-square test completed
- [ ] Independent t-test completed
- [ ] Logistic Regression metrics + confusion matrix
- [ ] Decision Tree metrics + confusion matrix
- [ ] Random Forest metrics + confusion matrix
- [ ] Feature importance plot
- [ ] Business recommendations
- [ ] Public GitHub repository named `customer-churn-analysis`
- [ ] Optional 2-minute screen recording
